In [16]:
import pandas as pd
import numpy as np

In [17]:
# --- 1. SETUP ----
MAIN_DATA_FILE = 'C:/Users/raffi/OneDrive - United Nations/Desktop/DSS/DATA AVAILABILITY INTERACTIVE DASHBOARD/final_dataset_combined.xlsx'
CRITERIA_FILE = 'C:/Users/raffi/OneDrive - United Nations/Desktop/DSS/DATA AVAILABILITY INTERACTIVE DASHBOARD/criterias.xlsx'

# Load data
main_df = pd.read_excel(MAIN_DATA_FILE)
criteria_df = pd.read_excel(CRITERIA_FILE).dropna(subset=['Indicator_Ar'])[['Indicator_Ar', 'criteria']]

# Force numeric values (Invalid text becomes NaN)
main_df['val_numeric'] = pd.to_numeric(main_df['العدد'], errors='coerce')
# Convert Year to numeric, turning any non-year text into NaN
main_df['السنة'] = pd.to_numeric(main_df['السنة'], errors='coerce')

In [18]:
# --- 2. GENERAL AVAILABILITY ---

# 2. Define conditions based on the CURRENT dataframe being processed
# Move these inside the logic so they apply to main_df before it's grouped
conds = [
    (main_df['السنة'] >= 2010) & (main_df['السنة'] <= 2014),
    (main_df['السنة'] >= 2015) & (main_df['السنة'] <= 2019),
    (main_df['السنة'] >= 2020) & (main_df['السنة'] <= 2026)
]
bins = ['2010-2014', '2015-2019', '2020-2026']

# 3. Add the bracket to the full dataframe FIRST
main_df['bracket'] = np.select(conds, bins, default='Other')

# 4. Now perform the collapse. Include 'bracket' in the groupby so you don't lose it.
df_gen = main_df[main_df['bracket'] != 'Other'].copy()
df_gen = df_gen.groupby(['المؤشر', 'الدولة', 'السنة', 'bracket'])['val_numeric'].max().reset_index()

# 5. Add valid data flag
df_gen['is_valid'] = df_gen['val_numeric'].notna().astype(int)

# 6. Sum valid years per bracket
gen_brk = df_gen.groupby(['المؤشر', 'الدولة', 'bracket'])['is_valid'].sum().reset_index()

# 7. Calculate Group Size
gen_brk['group_size'] = gen_brk.groupby(['المؤشر', 'الدولة'])['bracket'].transform('size')

# 8. Merge Criteria and Apply Logic
gen_final = pd.merge(gen_brk, criteria_df, left_on='المؤشر', right_on='Indicator_Ar', how='left')

# Availability = 1 if (group_size is 3) AND (every bracket count >= criteria)
gen_final['is_ok'] = ((gen_final['group_size'] == 3) & (gen_final['is_valid'] >= gen_final['criteria'])).astype(int)

# 9. Final Collapse
final_gen_report = gen_final.groupby(['المؤشر', 'الدولة'])['is_ok'].min().reset_index(name='general_availability')
final_gen_report.to_excel('general_availability.xlsx', index=False)

In [19]:
# --- 3. NATIONALITY AVAILABILITY ---

# A. Initial Clean 
# (Note: main_df already has the 'bracket' column from the General Availability step)
df_nat = main_df.dropna(subset=['المواطنة', 'val_numeric'])
df_nat = df_nat[~df_nat['المواطنة'].isin(['Not applicable', 'غير متاح', 'Nationality Total','مجموع المواطنين وغير المواطنين'])]

# B. Collapse to unique years (Checking if EITHER has data)
# We include 'bracket' in the groupby so it carries over automatically
df_nat_yearly = df_nat.groupby(['المؤشر', 'الدولة', 'السنة', 'bracket'])['val_numeric'].max().reset_index()

# Filter out 'Other' brackets (already grouped, so length is consistent)
df_nat_yearly = df_nat_yearly[df_nat_yearly['bracket'] != 'Other']

# C. Count years in bracket and get Group Size
nat_brk = df_nat_yearly.groupby(['المؤشر', 'الدولة', 'bracket']).size().reset_index(name='year_count')
nat_brk['group_size'] = nat_brk.groupby(['المؤشر', 'الدولة'])['bracket'].transform('size')

# D. Merge Criteria and Apply Logic
nat_final = pd.merge(nat_brk, criteria_df, left_on='المؤشر', right_on='Indicator_Ar', how='left')
nat_final['is_ok'] = ((nat_final['group_size'] == 3) & (nat_final['year_count'] >= nat_final['criteria'])).astype(int)

# E. Final Collapse
final_nat_report = nat_final.groupby(['المؤشر', 'الدولة'])['is_ok'].min().reset_index(name='nationality_availability')
final_nat_report.to_excel('nationality_availability.xlsx', index=False)

In [20]:
# --- 4. SEX AVAILABILITY ---

# A. Remove blanks and filter categories
# df_sex inherits the 'bracket' column from main_df
df_sex = main_df.dropna(subset=['الجنس', 'val_numeric'])
df_sex = df_sex[~df_sex['الجنس'].isin(['Not applicable', 'غير متاح', 'Both sexes','كلا الجنسين'])]

# B. Collapse to Year
# Include 'bracket' in the groupby to carry it through
sex_yearly = df_sex.groupby(['المؤشر', 'الدولة', 'السنة', 'bracket'])['val_numeric'].max().reset_index()

# C. Sum years in bracket
# Filter for valid brackets and count years
sex_brk = sex_yearly[sex_yearly['bracket'] != 'Other'].groupby(['المؤشر', 'الدولة', 'bracket']).size().reset_index(name='count')

# D. Merge Criteria and Apply Logic
sex_merged = pd.merge(sex_brk, criteria_df, left_on='المؤشر', right_on='Indicator_Ar', how='left')
sex_merged['is_met'] = (sex_merged['count'] >= sex_merged['criteria']).astype(int)

# E. Final Check (Must have 3 brackets and all criteria met)
final_sex = sex_merged.groupby(['المؤشر', 'الدولة']).filter(lambda x: len(x) == 3 and x['is_met'].all())
final_sex = final_sex[['المؤشر', 'الدولة']].drop_duplicates()
final_sex['sex_availability'] = 1

# F. Ensure all pairs are represented (filling missed criteria with 0)
all_pairs_sex = sex_merged[['المؤشر', 'الدولة']].drop_duplicates()
final_sex = pd.merge(all_pairs_sex, final_sex, on=['المؤشر', 'الدولة'], how='left').fillna(0)

final_sex.to_excel('sex_availability.xlsx', index=False)

print("Process Complete. 3 files saved.")

Process Complete. 3 files saved.


### creating the tables for the PBI TAG report

In [21]:
# --- 5. POWER BI TABLE 1: MAIN GRANULAR ---
# Ensure numeric and handle grouping
main_df['val_numeric'] = pd.to_numeric(main_df['العدد'], errors='coerce')
# Convert Year to numeric, turning any non-year text into NaN
main_df['السنة'] = pd.to_numeric(main_df['السنة'], errors='coerce')

# Grouping by Country, Indicator, Chapter (الفصل), and Year
pbi_main = main_df.groupby(['الدولة', 'المؤشر', 'الفصل', 'السنة'])['val_numeric'].max().reset_index()

# Availability flag: 1 if it's a number, 0 if NaN
pbi_main['available'] = pbi_main['val_numeric'].notna().astype(int)

pbi_main.to_excel('pbi_main_granular.xlsx', index=False)

In [22]:
# --- 6. POWER BI TABLE 2: NATIONALITY & SEX GRANULAR ---

# Nationality
#use pbi_nat from above
pbi_nat = df_nat.groupby(['الدولة', 'المؤشر', 'الفصل', 'السنة'])['val_numeric'].max().reset_index()
pbi_nat['available'] = pbi_nat['val_numeric'].notna().astype(int)
pbi_nat.to_excel('pbi_nationality_granular.xlsx', index=False)

# Sex
#use pbi_sex form above
pbi_sex = df_sex.groupby(['الدولة', 'المؤشر', 'الفصل', 'السنة'])['val_numeric'].max().reset_index()
pbi_sex['available'] = pbi_sex['val_numeric'].notna().astype(int)
pbi_sex.to_excel('pbi_sex_granular.xlsx', index=False)

In [23]:
# --- 7. SUMMARY PERCENTAGE TABLES (DIRECT LINEAR) ---

# A. Create a mapping of Indicators to Chapters and get Denominators
indicator_map = main_df[['المؤشر', 'الفصل']].drop_duplicates()
chapter_totals = indicator_map.groupby('الفصل')['المؤشر'].nunique().reset_index(name='total_in_chapter')

# B. GENERAL AVAILABILITY SUMMARY
# 1. Merge the chapter back in
report_gen_with_chapter = pd.merge(final_gen_report, indicator_map, on='المؤشر', how='left')
# 2. Group and calculate
summary_gen = report_gen_with_chapter.groupby(['الدولة', 'الفصل'])['general_availability'].sum().reset_index()
summary_gen = pd.merge(summary_gen, chapter_totals, on='الفصل', how='left')
summary_gen['pct'] = (summary_gen['general_availability'] / summary_gen['total_in_chapter']) * 100
summary_gen.to_excel('summary_pct_general.xlsx', index=False)

# C. NATIONALITY AVAILABILITY SUMMARY
# 1. Merge the chapter back in
report_nat_with_chapter = pd.merge(final_nat_report, indicator_map, on='المؤشر', how='left')
# 2. Group and calculate
summary_nat = report_nat_with_chapter.groupby(['الدولة', 'الفصل'])['nationality_availability'].sum().reset_index()
summary_nat = pd.merge(summary_nat, chapter_totals, on='الفصل', how='left')
summary_nat['pct'] = (summary_nat['nationality_availability'] / summary_nat['total_in_chapter']) * 100
summary_nat.to_excel('summary_pct_nationality.xlsx', index=False)

# D. SEX AVAILABILITY SUMMARY
# 1. Merge the chapter back in
report_sex_with_chapter = pd.merge(final_sex, indicator_map, on='المؤشر', how='left')
# 2. Group and calculate
summary_sex = report_sex_with_chapter.groupby(['الدولة', 'الفصل'])['sex_availability'].sum().reset_index()
summary_sex = pd.merge(summary_sex, chapter_totals, on='الفصل', how='left')
summary_sex['pct'] = (summary_sex['sex_availability'] / summary_sex['total_in_chapter']) * 100
summary_sex.to_excel('summary_pct_sex.xlsx', index=False)

print("Process Complete. Summary files saved with Chapter data.")

Process Complete. Summary files saved with Chapter data.
